In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/working'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip show torch torchvision torchaudio pytorch_lightning timm albumentations torchmetrics
# !pip install lig

In [ ]:
import json, random
from collections import defaultdict
import os
import io
import hashlib
import requests
from PIL import Image
from tqdm import tqdm
import statistics
import matplotlib.pyplot as plt
import random
import imagehash

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import pytorch_lightning as pl
import timm
import torch.nn as nn
import torch
# import lightly
# from lightly.loss.dino_loss import DINOLoss
import torch.nn.functional as F
from torch.utils.data import Sampler
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor, EarlyStopping
from timm.models.vision_transformer import vit_small_patch16_224
import copy

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch.distributed as dist

# Data Cleaning: Load images, hash, remove duplicates.

In [ ]:
# -------- Recursive Image Extractor --------
def extract_routes(data):
    """Recursively extract route images and metadata from hierarchical data."""
    routes = []
    if isinstance(data, dict):
        if "routes" in data:
            for r in data["routes"]:
                routes.append({
                    "route_url": r.get("url")
                    "route_name": r.get("name"),
                    "images": r.get("images", [])
                })
        for k, v in data.items():
            if isinstance(v, (dict, list)):
                routes.extend(extract_routes(v))
    elif isinstance(data, list):
        for item in data:
            routes.extend(extract_routes(item))
    return routes

In [ ]:
# tagged = json.load(open("/kaggle/input/ontariosouthbouldering/tagged_ontario_south_bouldering_and_rock_tree.json"))

class dataset_orchestrator:
    def __init__(self, json_path):
        self.json_path = json_path
        self.json = json.load(open(json_path))
        
    def group_imgs(self, identifier):
        # identifier is the key name which holds the name of the route identifier (in this case route)
        # group images by route (only kept)
        route2imgs = defaultdict(list)
        for url, meta in self.json.items():
            if meta["tag"] == "keep":
                route2imgs[meta[identifier]].append(url)
        return route2imgs

    def find_multiview_routes(self, route2imgs):
        return [r for r, imgs in route2imgs.items() if len(imgs) >= 2]

    def find_imgs(self, multiview_routes):
        multiview_imgs = [r for r, meta in self.json.items() if meta["tag"] == "keep" and meta["route"] in multiview_routes]
        singleview_imgs = [r for r, meta in self.json.items() if meta["tag"] == "keep" and meta["route"] not in multiview_routes]
        return singleview_imgs, multiview_imgs

path = "/kaggle/input/ontariosouthbouldering/tagged_ontario_south_bouldering_and_rock_tree.json"
orchestrator = dataset_orchestrator(path)
route2imgs = orchestrator.group_imgs("route")
multiview_routes = orchestrator.find_multiview_routes(route2imgs)
all_imgs_index = orchestrator.json
singleview_imgs, multiview_imgs = orchestrator.find_imgs(multiview_routes)
print("Single View Image count:", len(singleview_imgs))
print("Multi View Image count:", len(multiview_imgs))

In [ ]:
class ImageDownloader:
    def __init__(self):
        self.seen_hashes = {}
        
    def fname_from_url(self, url: str) -> str:
        h = hashlib.sha1(url.encode()).hexdigest()
        return f"{h}.jpg"

    def is_duplicate(self, img_hash, hash_threshold = 3):
        """Check if the hash is close to any existing image hash."""
        for h in self.seen_hashes.keys():
            if img_hash - h <= hash_threshold:
                return True, self.seen_hashes[h]   # return original match info
        return False, None

    def download_and_save(self, url, out_dir):
        fname = self.fname_from_url(url)
        path = os.path.join(out_dir, fname)
    
        if os.path.exists(path):
            # already downloaded -> still need hash check for duplicates
            try:
                img_hash = imagehash.phash(Image.open(path))
                dup, info = self.is_duplicate(img_hash)
                if not dup:
                    self.seen_hashes[img_hash] = {"path": path, "url": url}
                    return path
                else:
                    print(f"⚠ Duplicate skipped: {url} matches {info['url']}")
                    return None
            except:
                return None
    
        try:
            resp = requests.get(url, timeout=10)
            img = Image.open(io.BytesIO(resp.content)).convert("RGB")
            img_hash = imagehash.phash(img)
    
            dup, info = self.is_duplicate(img_hash)
            if dup:
                print(f"⚠ Duplicate skipped: {url} matches {info['url']}")
                return None
    
            img.save(path)
            self.seen_hashes[img_hash] = {"path": path, "url": url}
            return path
    
        except Exception as e:
            print(f"❌ Failed on {url}: {e}")
            return None

    def download_group(self, urls, all_imgs, out_dir):
        index = []
        for url in tqdm(urls):
            path = self.download_and_save(url, out_dir)
            if path is not None:
                index.append({"url": url, "path": path, "route": all_imgs[url]["route"]})
        return index
        
    def check_for_missing(self, index):
        # count number of images per route
        counts = defaultdict(int)
        for sample in index:
            counts[sample["route"]] += 1
        return counts

OUTPUT_DIR = "data"
SINGLEVIEW_DIR = os.path.join(OUTPUT_DIR, "singleview")
MULTIVIEW_DIR = os.path.join(OUTPUT_DIR, "multiview")

os.makedirs(SINGLEVIEW_DIR, exist_ok=True)
os.makedirs(MULTIVIEW_DIR, exist_ok=True)

downloader = ImageDownloader()
print("\n📥 Downloading SINGLEVIEW set...")
singleview_index = downloader.download_group(singleview_imgs, all_imgs_index, SINGLEVIEW_DIR)
print("\n📥 Downloading MULTIVIEW set...")
multiview_index = downloader.download_group(multiview_imgs, all_imgs_index, MULTIVIEW_DIR)

# Now drop any images without a partner from the multiview set and append them :
# from collections import defaultdict
counts = downloader.check_for_missing(multiview_index)

# filter out any route that has only one image
multiview_index = [s for s in multiview_index if counts[s["route"]] > 1]
extra_singles_index = [s for s in multiview_index if counts[s["route"]] == 1]
# Add what we took off of the multiview dataset to the singleviews
singleview_index.extend(extra_singles_index)

# --------------------------------------------------
# Save index files
# --------------------------------------------------
with open(os.path.join(OUTPUT_DIR, "index_simclr.json"), "w") as f:
    json.dump(singleview_index, f, indent=2)

with open(os.path.join(OUTPUT_DIR, "index_multiview.json"), "w") as f:
    json.dump(multiview_index, f, indent=2)

print("\n🎉 Done!")
print(f"SINGLEVIEW saved: {len(singleview_index)} images")
print(f"MULTIVIEW saved: {len(multiview_index)} images")
print(f"Data stored in: {OUTPUT_DIR}/")

# Create transformations and loaders

In [ ]:
# ------------------------------
# 1. Dataset: Multi-crop for DINO
# ------------------------------
class DinoDataset(Dataset):
    """
    Returns multiple crops per image:
    - 2 global crops (224x224)
    - N local crops (96x96)
    """
    def __init__(self, image_paths, N_local=6):
        self.paths = image_paths
        self.N_local = N_local
        
        # global crop (224x224) with outdoor realism
        self.global_transform = A.Compose([
            # A.Perspective(scale=(0.05,0.3)),
            # A.Resize(224,224),
            A.RandomResizedCrop((224, 224), scale=(0.6, 1.0)),
            A.ColorJitter(),
            A.GaussianBlur(3, sigma_limit=(0.1,2.0)),
            # A.RandomShadow(shadow_roi=(0,0.5,1,1), num_shadows_limit=[1,2], shadow_intensity_range=[0.1,0.8]),
            # A.RandomSunFlare(flare_roi=(0,0.5,1,1), angle_lower=0.5, angle_upper=1.5, num_flare_circles_lower=1, num_flare_circles_upper=3, src_radius=100),
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2()
        ])

        # local crops (96x96)
        self.local_transform = A.Compose([
            # A.Perspective(scale=(0.05,0.3)),
            # A.Resize(224,224),
            A.RandomResizedCrop((224, 224), scale=(0.25, 0.4)),
            A.ColorJitter(),
            A.GaussianBlur(3, sigma_limit=(0.1,2.0)),
            # A.RandomShadow(shadow_roi=(0,0.5,1,1), num_shadows_limit=[1,2], shadow_intensity_range=[0.1,0.8]),
            # A.RandomSunFlare(flare_roi=(0,0.5,1,1), angle_lower=0.5, angle_upper=1.5, num_flare_circles_lower=1, num_flare_circles_upper=3, src_radius=100),
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2()
        ])
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        path = self.paths[idx]
        img = np.array(Image.open(path).convert("RGB"))

        crops = [self.global_transform(image=img)["image"] for _ in range(2)]
        crops += [self.local_transform(image=img)["image"] for _ in range(self.N_local)]
        return crops

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# Example: take a few images from your dataset
example_paths = ["/kaggle/working/data/singleview/014c0e6610be2f7363f4d7c9ddc62ce17d2add31.jpg"]  # first 3 images

# Number of augmented versions to visualize per image
n_augmentations = 4

# Initialize your Albumentations dataset
aug_dataset = DinoDataset(example_paths, N_local=2)

for i, path in enumerate(example_paths):
    fig, axes = plt.subplots(2, n_augmentations, figsize=(15,5))
    img = np.array(Image.open(path).convert("RGB"))
    axes[0,i].imshow(img)
    axes[0,i].axis('off')
    axes[0,i].set_title(f"Original")
    
    for j in range(n_augmentations):
        # Apply one augmentation (global crop or local crop)
        aug_img = aug_dataset.global_transform(image=img)["image"]  # or local_transform
        aug_img = aug_img.permute(1,2,0).numpy()  # CHW -> HWC
        aug_img = np.clip(aug_img, 0, 1)
        
        axes[1,j].imshow(aug_img)
        axes[1,j].axis('off')
        axes[1,j].set_title(f"Aug {j+1}")
    
    plt.suptitle(f"Original: {path.split('/')[-1]}")
    plt.show()


# Create Model Training Architecture

In [ ]:
class DINOLoss(nn.Module):
    def __init__(self, out_dim, ncrops, warmup_teacher_temp, teacher_temp,
                warmup_teacher_temp_epochs, nepochs, student_temp = 0.1, center_momentum = 0.9):
        super().__init__()
        self.student_temp = student_temp
        self.center_momentum = center_momentum
        self.ncrops = ncrops
        self.register_buffer("center", torch.zeros(1, out_dim))
        # Create a warmup schedule for teacher to improve stability:
        self.teacher_temp_schedule = np.concatenate((
            np.linspace(warmup_teacher_temp, teacher_temp, warmup_teacher_temp_epochs),
            np.ones(nepochs - warmup_teacher_temp_epochs) * teacher_temp
        )) # goes like [warmup_temp... -> teacher temp...]

    def forward(self, student_output, teacher_output, epoch):
        # Loss module takes in teacher and student outputs and uses temp based on epoch and schedule
        student_out = student_output / self.student_temp
        student_out = student_out.chunk(self.ncrops)

        # Center + temperature to avoid representation collapse
        temp = self.teacher_temp_schedule[epoch]
        teacher_out = F.softmax((teacher_output - self.center) / temp, dim = -1) # softmax over the lastr dimension
        teacher_out = teacher_out.detach().chunk(2) # break up into 2 global views
        
        # Compute Cross Entropy between teacher and student outputs:
        total_loss = 0
        n_loss_terms = 0
        for iq, q in enumerate(teacher_out):
            for iv, v in enumerate(student_out):
                if iv == iq:
                    # Skip where student and teacher operate on the same global view
                    continue
                loss = torch.sum(-q * F.log_softmax(v, dim = -1), dim = -1) # Sum over last dimension
                # Compute mean of representation loss
                total_loss += loss.mean()
                n_loss_terms += 1
        total_loss /= n_loss_terms
        self.update_center(teacher_output)
        return total_loss

    @torch.no_grad()
    def update_center(self, teacher_output):
        batch_center = torch.sum(teacher_output, dim = 0, keepdim = True)
        # dist.all_reduce(batch_center)
        # batch_center = batch_center / len(teacher_output) * dist.get_world_size()
        #ema update
        self.center = self.center * self.center_momentum + batch_center * (1- self.center_momentum)
        

In [ ]:
# Coppied from Dino Repo
class MultiCropWrapper(nn.Module):
    """
    Perform forward pass separately on each resolution input.
    The inputs corresponding to a single resolution are clubbed and single
    forward is run on the same resolution inputs. Hence we do several
    forward passes = number of different resolutions used. We then
    concatenate all the output features and run the head forward on these
    concatenated features.
    """
    def __init__(self, backbone, head):
        super(MultiCropWrapper, self).__init__()
        # disable layers dedicated to ImageNet labels classification
        backbone.fc, backbone.head = nn.Identity(), nn.Identity()
        self.backbone = backbone
        self.head = head

    def forward(self, x):
        # convert to list
        if not isinstance(x, list):
            x = [x]
        idx_crops = torch.cumsum(torch.unique_consecutive(
            torch.tensor([inp.shape[-1] for inp in x]),
            return_counts=True,
        )[1], 0)
        start_idx, output = 0, torch.empty(0).to(x[0].device)
        for end_idx in idx_crops:
            _out = self.backbone(torch.cat(x[start_idx: end_idx]))
            # The output is a tuple with XCiT model. See:
            # https://github.com/facebookresearch/xcit/blob/master/xcit.py#L404-L405
            if isinstance(_out, tuple):
                _out = _out[0]
            # accumulate outputs
            output = torch.cat((output, _out))
            start_idx = end_idx
        # Run the head forward on the concatenated features.
        return self.head(output)

In [ ]:
# DINO trainer:
class DinoHead(nn.Module):
    def __init__(self, embed_dim, hid_dim=2048, bottleneck_dim=256, out_dim=65536):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hid_dim),
            # nn.BatchNorm1d(hid_dim),
            nn.GELU(),
            nn.Linear(hid_dim, bottleneck_dim),
        )
        self.last_layer = nn.utils.weight_norm(nn.Linear(bottleneck_dim, out_dim, bias=False))
        self.last_layer.weight_g.data.fill_(1)

    def forward(self, x):
        x = self.mlp(x)
        return F.normalize(self.last_layer(x), dim=-1)         

class DINOModule(pl.LightningModule):
    def __init__(self, loss_instance, model_name = "vit_small_patch16_224", 
                 pretrained = True, out_dim = 65536, lr = 1e-3, weight_decay = 0.05,
                 initial_teacher_momentum = 0.996, num_unfrozen_blocks = 2):
        super().__init__()
        self.save_hyperparameters()

        self.dino_loss = loss_instance
        self.initial_teacher_momentum = initial_teacher_momentum
        self.lr = lr
        self.weight_decay = weight_decay
        self.num_unfrozen_blocks = 2
        
        # Create teacher/student
        teacher = timm.create_model(model_name, pretrained=pretrained)
        student = copy.deepcopy(teacher)
        self.embed_dim = teacher.embed_dim
        
        # Wrap for multicrop forward pass
        self.teacher = MultiCropWrapper(teacher, DinoHead(self.embed_dim, out_dim = out_dim))
        self.student = MultiCropWrapper(student, DinoHead(self.embed_dim, out_dim = out_dim))

        # finetuning by freezing most blocks
        self.freeze_blocks()
        
        # Freeze teacher weights
        for p in self.teacher.parameters():
            p.requires_grad = False

    def freeze_blocks(self):
        all_blocks = len(self.student.backbone.blocks)
        for i in range(0, all_blocks - self.num_unfrozen_blocks):
            for param in self.student.backbone.blocks[i].parameters():
                param.requires_grad = False
        
    def on_train_start(self):
        self.momentum_schedule = np.linspace(self.hparams.initial_teacher_momentum, 1, self.trainer.max_epochs)

    def on_train_batch_end(self, outputs, batch, batch_idx):
        # IMPORTANT: this runs after backward + optimizer step
        with torch.no_grad():
            m = self.momentum_schedule[self.current_epoch]  # or compute momentum dynamically
            for student_p, teacher_p in zip(self.student.parameters(), self.teacher.parameters()):
                teacher_p.data = m * teacher_p.data + (1 - m) * student_p.data
        
    def training_step(self, batch, batch_idx):
        self.teacher.eval()
        # if self.current_epoch % 5 == 0:
        #     teacher_norm = sum(p.norm().item() for p in self.teacher.parameters())
        #     student_norm = sum(p.norm().item() for p in self.student.parameters())
        #     print(f"[DEBUG] epoch = {self.current_epoch}, teacher norm = {teacher_norm:.2f}, student norm = {student_norm:.2f}")

        # Process each set of image crops within batch
        student_out = self.student(batch)
        teacher_out = self.teacher(batch[:2])
        train_loss = self.dino_loss(student_out, teacher_out, self.current_epoch)
        self.log("train_loss", train_loss, sync_dist=True)
        # update center:
        # self.dino_loss.update_center(teacher_out)
        return train_loss 
        
    @torch.no_grad()
    def validation_step(self, batch, batch_idx):
        # Process each set of image crops within batch
        # Process each set of image crops within batch
        student_out = self.student(batch)
        teacher_out = self.teacher(batch[:2])
        val_loss = self.dino_loss(student_out, teacher_out, self.current_epoch)
        self.log("val_loss", val_loss, sync_dist=True)
        return val_loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.student.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=300)
        return [optimizer], [scheduler]

In [ ]:
# Hyper parameters
output_dim = 1024
n_locals = 8
n_crops = n_locals + 2

warmup_teacher_temp = 0.08
teacher_temp = 0.1
warmup_teacher_temp_epochs = 10
student_temp = 0.1
center_momentum = 0.9
bs = 16
lr =  0.0005 * bs/256
weight_decay = 0.01
initial_teacher_momentum = 0.996
num_unfrozen_blocks = 0

nepochs = 100

# Create Data Loaders
singleview_paths = [s["path"] for s in singleview_index]  # list of paths
multiview_paths = [s["path"] for s in multiview_index]
train_ds = DinoDataset(singleview_paths, N_local = n_locals)
val_ds = DinoDataset(multiview_paths, N_local = n_locals)

train_loader = DataLoader(train_ds, batch_size = bs, num_workers = 4, shuffle = True)
val_loader = DataLoader(val_ds, batch_size = bs, num_workers = 4, shuffle = False)

# Create Loss Instance
loss_instance = DINOLoss(out_dim = output_dim, ncrops = n_crops, 
                         warmup_teacher_temp = warmup_teacher_temp, 
                         teacher_temp = teacher_temp, 
                         warmup_teacher_temp_epochs = warmup_teacher_temp_epochs,
                         nepochs = nepochs, student_temp = student_temp,
                         center_momentum = center_momentum)

# Setup Callbacks:
checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    save_top_k=1,
    mode="min",
    filename="dino-{epoch:02d}-{train_loss:.2f}-{val_loss:.2f}"
)

early_stop_callback = EarlyStopping(
    monitor="val_loss",     # metric to monitor
    patience=10,            # stop after 10 epochs with no improvement
    verbose=True,
    mode="min"              # lower val_loss is better
)

lr_monitor = LearningRateMonitor(logging_interval="step")

# Trainer
trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=nepochs,
    precision=16,
    log_every_n_steps=1,
    callbacks=[early_stop_callback, checkpoint_callback, lr_monitor],
)

# ------------------------------
# 5. Start training
# ------------------------------
model = DINOModule(loss_instance = loss_instance,
                  out_dim = output_dim,
                  lr = lr, 
                  weight_decay = weight_decay,
                  initial_teacher_momentum = initial_teacher_momentum,
                  num_unfrozen_blocks = num_unfrozen_blocks)

trainer.fit(model, train_loader, val_loader)

In [ ]:
import shutil
import zipfile
from IPython.display import FileLink

# Download Model:
best_ckpt = checkpoint_callback.best_model_path
# name = best_ckpt.split("/")[-1]
# print("Best checkpoint:", best_ckpt)
# zip_path = "/kaggle/working/dino_v1.zip"

# with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
#     z.write(best_ckpt, arcname=name)

# print("Zipped checkpoint saved to:", zip_path)

# FileLink(zip_path.split("/")[-1])

In [ ]:
# Load Model
model = DINOModule.load_from_checkpoint(best_ckpt)
_ = model.eval()
encoder = model.teacher

In [ ]:
# --- 1. Define transforms for evaluation (no random augmentation!) ---
EVAL_TRANSFORMS = T.Compose([
    T.Resize((224, 224)),      # or whatever you prefer
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

# --- 2. Generate embeddings without DataLoader ---
# encoder = timm.create_model("vit_small_patch16_224", pretrained=True).to("cuda")
encoder.eval()
all_embeddings = []
all_paths = []
all_routes = []

with torch.no_grad():
    for sample in multiview_index:   # each sample is a dict {"path":..., "route":...}
        img = Image.open(sample["path"]).convert("RGB")
        x = EVAL_TRANSFORMS(img).unsqueeze(0)#.to(device)  # add batch dim
        z = encoder(x.cuda())
        all_embeddings.append(z.cpu())
        all_paths.append(sample["path"])
        all_routes.append(sample["route"])

embeddings = torch.cat(all_embeddings)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

# KNN
knn = NearestNeighbors(n_neighbors=4, metric='cosine')
knn.fit(embeddings)
distances, indices = knn.kneighbors(embeddings)

# Fraction top-1 / top-3 matches
match_count = sum([all_routes[i] == all_routes[indices[i][1]] for i in range(len(all_routes))])
print(f"Top-1 match fraction: {match_count/len(all_routes)*100:.2f}%")
top3_matches = sum([all_routes[i] in [all_routes[j] for j in indices[i][1:]] for i in range(len(all_routes))])
print(f"Top-3 match fraction: {top3_matches/len(all_routes)*100:.2f}%")

In [ ]:
def plot_knn_from_index(paths, routes, indices, n_samples=10):
    import random
    sampled_idxs = random.sample(range(len(paths)), min(n_samples, len(paths)))
    
    for anchor_idx in sampled_idxs:
        anchor_img = Image.open(paths[anchor_idx]).convert("RGB")
        pred_idx = indices[anchor_idx][1]  # top-1 neighbor
        pred_img = Image.open(paths[pred_idx]).convert("RGB")
        
        # actual match that's not the anchor
        actual_matches = [i for i, r in enumerate(routes) if r == routes[anchor_idx] and i != anchor_idx]
        actual_idx = actual_matches[0] if actual_matches else None
        
        n_cols = 3 if actual_idx and actual_idx != pred_idx else 2
        fig, axs = plt.subplots(1, n_cols, figsize=(4*n_cols, 4))
        
        axs[0].imshow(anchor_img)
        axs[0].set_title(f"Anchor\nRoute {routes[anchor_idx]}")
        axs[0].axis('off')
        
        axs[1].imshow(pred_img)
        axs[1].set_title(f"Top-1 Pred\nRoute {routes[pred_idx]}")
        axs[1].axis('off')
        
        if n_cols == 3:
            actual_img = Image.open(paths[actual_idx]).convert("RGB")
            axs[2].imshow(actual_img)
            axs[2].set_title(f"Actual Match\nRoute {routes[actual_idx]}")
            axs[2].axis('off')
        
        plt.show()

plot_knn_from_index(all_paths, all_routes, indices, n_samples=30)